# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
**Profesor**: Pablo Camarillo Ramirez

## Jesús Alejandro Toral Iñiguez

In [1]:
import sys
import os

notebook_path = os.path.abspath('')
module_path = os.path.abspath(os.path.join(notebook_path, '../../src'))

from AlexTModule.spark_utils import SparkUtils


spark_url = "local[*]"
app_name = "Lab 04"
su = SparkUtils(app_name, spark_url)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 00:51:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 00:51:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
## Agencies Dataframe

columns_types = [
    ("agency_id", "int"),
    ("agency_info","string")

]

agency_schema = SparkUtils.generate_schema(columns_types)

agency_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(agency_schema) \
                .csv("/opt/spark/work-dir/data/car_service/agencies/agencies.csv")



In [3]:
## Brands Dataframe

columns_types = [
    ("brand_id", "int"),
    ("brand_info","string")

]

brand_schema = SparkUtils.generate_schema(columns_types)

brand_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(brand_schema) \
                .csv("/opt/spark/work-dir/data/car_service/brands/brands.csv")




In [4]:
## Cars Dataframe

columns_types = [
    ("car_id", "int"),
    ("car_info","string")

]

car_schema = SparkUtils.generate_schema(columns_types)

car_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(car_schema) \
                .csv("/opt/spark/work-dir/data/car_service/cars/cars.csv")



In [5]:
## Customers Dataframe

columns_types = [
    ("customer_id", "int"),
    ("customer_info","string")

]

customer_schema = SparkUtils.generate_schema(columns_types)

customer_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(customer_schema) \
                .csv("/opt/spark/work-dir/data/car_service/customers/customers.csv")



In [6]:

## rentals Dataframe

columns_types = [
    ("rental_id", "int"),
    ("rental_info","string")

]

rentals_schema = SparkUtils.generate_schema(columns_types)

rentals_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(rentals_schema) \
                .csv("/opt/spark/work-dir/data/car_service/rentals/")




In [7]:
from pyspark.sql.functions import get_json_object, col
## JOINS

joined_df = rentals_df \
    .join(car_df,get_json_object(col("rental_info"),"$.car_id")==col("car_id")) \
    .join(customer_df,get_json_object(col("rental_info"),"$.customer_id")==col("customer_id")) \
    .join(agency_df,get_json_object(col("rental_info"),"$.agency_id")==col("agency_id"))


## FINAL SELECTS
final_df = joined_df.select(
    col("rental_id"),get_json_object(col("car_info"),"$.car_name").alias("car_name"),
        get_json_object(col("agency_info"),"$.agency_name").alias("agency_name"),
        get_json_object(col("customer_info"),"$.customer_name").alias("customer_name"),
)
final_df.show(5)

+---------+--------------------+-------------+---------------+
|rental_id|            car_name|  agency_name|  customer_name|
+---------+--------------------+-------------+---------------+
|    11891|Wallace-Carlson M...|  NYC Rentals| Margaret Jones|
|    11892|Grimes-Green Model 8|LA Car Rental|Albert Williams|
|    11893|Stewart-Allen Mod...|      SF Cars|  Caleb Fleming|
|    11894|  Campos PLC Model 4|  NYC Rentals|  Andrew Butler|
|    11895|  Wagner LLC Model 1|      SF Cars|  Kristin Potts|
+---------+--------------------+-------------+---------------+
only showing top 5 rows
